# Sequence modeling and text generation with Neural Networks: Seq2Seq Architecture and Neural Machine Translation

This notebook contains **4 short tasks** for the lab session practice. Initially indicative examples are provided and then you are requested to proceed with your own implementations.

### Learning goals
- understand how text is turned into sequences of integers
- prepare simple input-target pairs for next-word prediction
- inspect the main parts of a Seq2Seq model
- run greedy decoding on a tiny toy translation example

### Language idea
The worked examples use **English** and **Dutch** so that the flow is easy to follow. In the tasks, you can keep the given data or **try your own Spanish examples**.

### Things to bear in mind
- Keep the examples **small and simple**
- Focus on understanding the **flow**
- It is fine if the toy model is not perfect

In [16]:
import torch
import torch.nn as nn
import torch.optim as optim

device = "cuda" if torch.cuda.is_available() else "cpu" #device and gpu check
print("Using device:", device)

# Tiny toy corpus for next-word prediction (English)
sentences = [
    "i like neural networks",
    "i like deep learning",
    "you like machine learning",
    "we study sequence models",
    "we study neural translation"
]

# Tiny toy translation pairs for the Lab Assignmen Tasks (English -> Spanish)
pairs = [
    ("i am happy", "yo estoy feliz"),
    ("i am sad", "yo estoy triste"),
    ("you are happy", "tu estas feliz"),
    ("you are sad", "tu estas triste")
]


Using device: cpu


### Indicative example before Task 1

Below is a **complete mini-example** using a tiny English corpus.

Read it first to understand the flow:

1. split text into words  
2. build a vocabulary  
3. map words to integers  
4. create `(context, target)` next-word examples

**Mini-example sentence:** `i learn today`
- context = `["i"]`, target = `"learn"`
- context = `["i", "learn"]`, target = `"today"`

After this example, you are requested to apply the same approach in the task.


In [8]:
# tokenization and next-word pairs
mini_sentences = [
    "Jack wants to play football during the weekend",
    "Emily has plans for dinner tonight"
]

mini_words = []
for s in mini_sentences:
    mini_words.extend(s.split())

mini_vocab = sorted(set(mini_words))
mini_word2idx = {w: i+1 for i, w in enumerate(mini_vocab)}  # 0 reserved for padding
mini_idx2word = {i: w for w, i in mini_word2idx.items()}

print("Mini vocabulary:", mini_vocab)
print("word2idx:", mini_word2idx)

mini_tokenized = []
for s in mini_sentences:
    ids = [mini_word2idx[w] for w in s.split()]
    mini_tokenized.append(ids)

print("\nTokenized mini sentences:")
for s, ids in zip(mini_sentences, mini_tokenized):
    print(s, "->", ids)

mini_examples = []
for ids in mini_tokenized:
    for i in range(1, len(ids)):
        context = ids[:i]
        target = ids[i]
        mini_examples.append((context, target))

print("\nMini training pairs:")
for context, target in mini_examples:
    context_words = [mini_idx2word[i] for i in context]
    target_word = mini_idx2word[target]
    print("context =", context_words, "-> target =", target_word)


Mini vocabulary: ['Emily', 'Jack', 'dinner', 'during', 'football', 'for', 'has', 'plans', 'play', 'the', 'to', 'tonight', 'wants', 'weekend']
word2idx: {'Emily': 1, 'Jack': 2, 'dinner': 3, 'during': 4, 'football': 5, 'for': 6, 'has': 7, 'plans': 8, 'play': 9, 'the': 10, 'to': 11, 'tonight': 12, 'wants': 13, 'weekend': 14}

Tokenized mini sentences:
Jack wants to play football during the weekend -> [2, 13, 11, 9, 5, 4, 10, 14]
Emily has plans for dinner tonight -> [1, 7, 8, 6, 3, 12]

Mini training pairs:
context = ['Jack'] -> target = wants
context = ['Jack', 'wants'] -> target = to
context = ['Jack', 'wants', 'to'] -> target = play
context = ['Jack', 'wants', 'to', 'play'] -> target = football
context = ['Jack', 'wants', 'to', 'play', 'football'] -> target = during
context = ['Jack', 'wants', 'to', 'play', 'football', 'during'] -> target = the
context = ['Jack', 'wants', 'to', 'play', 'football', 'during', 'the'] -> target = weekend
context = ['Emily'] -> target = has
context = ['Emil


## Task 1 — Tokenize the corpus and create next-word examples

Your goal is to transform words into integers and build simple training pairs.

### What to do
1. Build a vocabulary from `sentences`
2. Create a `word2idx` dictionary
3. Convert each sentence into token IDs
4. From each sentence, create simple `(context, target)` pairs for next-word prediction


In [17]:

# TODO: Build the vocabulary
all_words = []
for s in sentences:
    all_words.extend(s.split())
    pass

vocab = sorted(set(all_words))
word2idx = {w: i+1 for i, w in enumerate(vocab)}   # reserve 0 for padding
idx2word = {i: w for w, i in word2idx.items()}

print("Vocabulary:", vocab)
print("Vocabulary size:", len(vocab))

# TODO: Convert sentences to token IDs
tokenized_sentences = []
for s in sentences:
    ids = [word2idx[w] for w in s.split()]
    tokenized_sentences.append(ids)
    

print("\nTokenized sentences:")
for ts in tokenized_sentences:
    print(ts)

# TODO: Create (context, target) pairs
examples = []
for ts in tokenized_sentences:
    for i in range(1, len(ts)):
        context = ts[:i]
        target = ts[i]
        examples.append((context, target))
    

print("\nFirst 8 training examples:")
for x, y in examples[:8]:
    x = [idx2word[i] for i in x]
    y = idx2word[y]
    print("context =", x, "target =", y)


Vocabulary: ['deep', 'i', 'learning', 'like', 'machine', 'models', 'networks', 'neural', 'sequence', 'study', 'translation', 'we', 'you']
Vocabulary size: 13

Tokenized sentences:
[2, 4, 8, 7]
[2, 4, 1, 3]
[13, 4, 5, 3]
[12, 10, 9, 6]
[12, 10, 8, 11]

First 8 training examples:
context = ['i'] target = like
context = ['i', 'like'] target = neural
context = ['i', 'like', 'neural'] target = networks
context = ['i'] target = like
context = ['i', 'like'] target = deep
context = ['i', 'like', 'deep'] target = learning
context = ['you'] target = like
context = ['you', 'like'] target = machine


### Indicative example before Task 2

Now you see a **complete tiny neural next-word model**.

Flow:
1. represent each word with an **embedding**
2. average the embeddings in the context
3. use a **linear layer** to predict the next word
4. train for a few epochs

This is a very simple baseline model, just to understand the pipeline.


In [18]:
# tokenization and next-word pairs
mini_sentences = [
    "Jack wants to play football during the weekend",
    "Emily has plans for dinner tonight"
]

mini_words = []
for s in mini_sentences:
    mini_words.extend(s.split())
mini_vocab = sorted(set(mini_words))
mini_word2idx = {w: i+1 for i, w in enumerate(mini_vocab)}

mini_tokenized = [[mini_word2idx[w] for w in s.split()] for s in mini_sentences]

mini_examples = []
for ids in mini_tokenized:
    for i in range(1, len(ids)):
        mini_examples.append((ids[:i], ids[i]))

mini_max_len = max(len(ctx) for ctx, _ in mini_examples)
mini_X, mini_Y = [], []
for ctx, target in mini_examples:
    padded = ctx + [0] * (mini_max_len - len(ctx))
    mini_X.append(padded)
    mini_Y.append(target)

mini_X = torch.tensor(mini_X, dtype=torch.long)
mini_Y = torch.tensor(mini_Y, dtype=torch.long)

class MiniNextWordModel(nn.Module):
    def __init__(self, vocab_size, emb_dim=8):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size + 1, emb_dim, padding_idx=0)
        self.fc = nn.Linear(emb_dim, vocab_size + 1)

    def forward(self, x):
        emb = self.embedding(x)           # [batch, time, emb_dim]
        context_vec = emb.mean(dim=1)     # [batch, emb_dim]
        return self.fc(context_vec)       # [batch, vocab_size+1]

mini_model = MiniNextWordModel(len(mini_vocab))
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(mini_model.parameters(), lr=0.05)

for epoch in range(30):
    optimizer.zero_grad()
    logits = mini_model(mini_X)
    loss = criterion(logits, mini_Y)
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch}: loss = {loss.item():.4f}")

with torch.no_grad():
    preds = mini_model(mini_X).argmax(dim=1)

print("\nMini predictions:")
for x, y_true, y_pred in zip(mini_X, mini_Y, preds):
    ctx_words = [mini_vocab[idx-1] for idx in x.tolist() if idx != 0]
    print(ctx_words, "-> true:", mini_vocab[y_true.item()-1], "| pred:", mini_vocab[y_pred.item()-1])


Epoch 0: loss = 2.8059
Epoch 10: loss = 1.8714
Epoch 20: loss = 1.1674

Mini predictions:
['Jack'] -> true: wants | pred: wants
['Jack', 'wants'] -> true: to | pred: to
['Jack', 'wants', 'to'] -> true: play | pred: play
['Jack', 'wants', 'to', 'play'] -> true: football | pred: football
['Jack', 'wants', 'to', 'play', 'football'] -> true: during | pred: during
['Jack', 'wants', 'to', 'play', 'football', 'during'] -> true: the | pred: the
['Jack', 'wants', 'to', 'play', 'football', 'during', 'the'] -> true: weekend | pred: weekend
['Emily'] -> true: has | pred: has
['Emily', 'has'] -> true: plans | pred: plans
['Emily', 'has', 'plans'] -> true: for | pred: for
['Emily', 'has', 'plans', 'for'] -> true: dinner | pred: dinner
['Emily', 'has', 'plans', 'for', 'dinner'] -> true: tonight | pred: tonight



## Task 2 — Build a very small next-word model

Your goals is to use an embedding layer and a simple average over the context words.

This is **not** a full RNN yet. It is just a very simple neural baseline for next-word prediction.

### What to do
1. Pad contexts to the same length
2. Build a small model:
   - `Embedding`
   - average embeddings over the time dimension
   - `Linear` layer to predict the next word
3. Train for a few epochs
4. Print the loss


In [ ]:

# Prepare padded tensors
max_len = max(len(ctx) for ctx, _ in examples)

X = []
Y = []
for ctx, target in examples:
    padded = ctx + [0] * (max_len - len(ctx))
    X.append(padded)
    Y.append(target)

X = torch.tensor(X, dtype=torch.long).to(device)
Y = torch.tensor(Y, dtype=torch.long).to(device)

print("X shape:", X.shape)
print("Y shape:", Y.shape)

class SimpleNextWordModel(nn.Module):
    def __init__(self, vocab_size, emb_dim=16):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size + 1, emb_dim, padding_idx=0)
        self.fc = nn.Linear(emb_dim, vocab_size + 1)

    def forward(self, x):
        emb = self.embedding(x)                       # [batch, seq_len, emb_dim]

        mask = None

        # TODO: compute the average embedding over real tokens only
        avg_emb = None

        logits = self.fc(avg_emb)
        return logits

model = SimpleNextWordModel(len(vocab)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.02)

# TODO: train for 100 epochs
for epoch in range(100):
    optimizer.zero_grad()

    # TODO
    logits = None
    loss = None

    # TODO
    # loss.backward()
    # optimizer.step()

    if epoch % 20 == 0:
        print(f"Epoch {epoch}: loss = {loss.item():.4f}")


### Indicative example before Task 3

Before reading the larger Seq2Seq task, here is the **basic encoder-decoder flow** in one tiny example.

Idea:
- the **encoder** reads the English source sentence
- the **decoder** receives the encoder hidden state
- the decoder predicts Dutch output one step at a time

In this example, we only run **one forward pass** and inspect the tensor shapes.

After that, in the student task, you will implement the same logic for **English -> Spanish**.


In [19]:
# tiny encoder-decoder forward pass (English -> Dutch)
mini_pairs = [
    ("i am happy", "ik ben blij"),
    ("you are sad", "jij bent verdrietig")
]

mini_src_words = sorted(set(" ".join(src for src, tgt in mini_pairs).split()))
mini_tgt_words = sorted(set(" ".join(tgt for src, tgt in mini_pairs).split()))

mini_src2idx = {w: i+2 for i, w in enumerate(mini_src_words)}  # 0=PAD, 1=SOS
mini_tgt2idx = {w: i+2 for i, w in enumerate(mini_tgt_words)}  # 0=PAD, 1=SOS

def mini_encode_src(sentence):
    return [mini_src2idx[w] for w in sentence.split()]

def mini_encode_tgt(sentence):
    return [1] + [mini_tgt2idx[w] for w in sentence.split()]   # prepend SOS

mini_src = torch.tensor([mini_encode_src("i am happy")], dtype=torch.long)
mini_tgt = torch.tensor([mini_encode_tgt("ik ben blij")], dtype=torch.long)

class MiniEncoder(nn.Module):
    def __init__(self, vocab_size, emb_dim=8, hid_dim=16):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size + 2, emb_dim, padding_idx=0)
        self.gru = nn.GRU(emb_dim, hid_dim, batch_first=True)

    def forward(self, x):
        emb = self.embedding(x)
        outputs, hidden = self.gru(emb)
        return outputs, hidden

class MiniDecoder(nn.Module):
    def __init__(self, vocab_size, emb_dim=8, hid_dim=16):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size + 2, emb_dim, padding_idx=0)
        self.gru = nn.GRU(emb_dim, hid_dim, batch_first=True)
        self.fc = nn.Linear(hid_dim, vocab_size + 2)

    def forward(self, x, hidden):
        emb = self.embedding(x)
        outputs, hidden = self.gru(emb, hidden)
        logits = self.fc(outputs)
        return logits, hidden

mini_encoder = MiniEncoder(len(mini_src_words))
mini_decoder = MiniDecoder(len(mini_tgt_words))

mini_enc_outputs, mini_hidden = mini_encoder(mini_src)
mini_logits, mini_dec_hidden = mini_decoder(mini_tgt[:, :-1], mini_hidden)

print("Source sentence:", "i am happy")
print("Target sentence:", "ik ben blij")
print("Encoder outputs shape:", mini_enc_outputs.shape)
print("Encoder hidden shape:", mini_hidden.shape)
print("Decoder input shape:", mini_tgt[:, :-1].shape)
print("Decoder logits shape:", mini_logits.shape)


Source sentence: i am happy
Target sentence: ik ben blij
Encoder outputs shape: torch.Size([1, 3, 16])
Encoder hidden shape: torch.Size([1, 1, 16])
Decoder input shape: torch.Size([1, 3])
Decoder logits shape: torch.Size([1, 3, 8])


## Task 3 — Inspect a simple Seq2Seq architecture

Your goal understand the flow for **English -> Spanish**:

- encoder reads the source sentence
- decoder predicts the target sentence step by step

Below you have a tiny encoder-decoder model with GRUs.

### What to do
1. Read the code
2. Complete the `TODO` lines
3. Run one forward pass
4. Check the tensor shapes

**Nice to have**: Try to replace the GRUs with LSTMs

In [ ]:

# Build toy vocabularies for translation
src_words = sorted(set(" ".join(src for src, tgt in pairs).split()))
tgt_words = sorted(set(" ".join(tgt for src, tgt in pairs).split()))

src2idx = {w: i+2 for i, w in enumerate(src_words)}  # 0=PAD, 1=SOS
tgt2idx = {w: i+2 for i, w in enumerate(tgt_words)}  # 0=PAD, 1=SOS
idx2tgt = {i: w for w, i in tgt2idx.items()}
idx2tgt[1] = "<SOS>"

def encode_src(sentence):
    return [src2idx[w] for w in sentence.split()]

def encode_tgt(sentence):
    return [1] + [tgt2idx[w] for w in sentence.split()]  # prepend SOS

src_batch = [encode_src(src) for src, tgt in pairs]
tgt_batch = [encode_tgt(tgt) for src, tgt in pairs]

max_src = max(len(x) for x in src_batch)
max_tgt = max(len(x) for x in tgt_batch)

src_batch = [x + [0]*(max_src-len(x)) for x in src_batch]
tgt_batch = [x + [0]*(max_tgt-len(x)) for x in tgt_batch]

src_batch = torch.tensor(src_batch, dtype=torch.long).to(device)
tgt_batch = torch.tensor(tgt_batch, dtype=torch.long).to(device)

class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim=16, hid_dim=32):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size + 2, emb_dim, padding_idx=0)
        self.gru = nn.GRU(emb_dim, hid_dim, batch_first=True)

    def forward(self, x):
        # TODO
        emb = None
        outputs, hidden = None, None
        return outputs, hidden

class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim=16, hid_dim=32):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size + 2, emb_dim, padding_idx=0)
        self.gru = nn.GRU(emb_dim, hid_dim, batch_first=True)
        self.fc = nn.Linear(hid_dim, vocab_size + 2)

    def forward(self, x, hidden):
        # x shape: [batch, 1]
        # TODO
        emb = None
        output, hidden = None, None
        logits = None
        return logits, hidden

encoder = Encoder(len(src_words)).to(device)
decoder = Decoder(len(tgt_words)).to(device)

enc_outputs, enc_hidden = encoder(src_batch)
print("Encoder outputs shape:", enc_outputs.shape)
print("Encoder hidden shape:", enc_hidden.shape)

decoder_input = tgt_batch[:, 0].unsqueeze(1)   # first token = SOS
logits, dec_hidden = decoder(decoder_input, enc_hidden)

print("Decoder logits shape:", logits.shape)
print("Decoder hidden shape:", dec_hidden.shape)


### Indicative example before Task 4

Here is a **complete greedy decoding example** for a tiny **English -> Dutch** toy case.

Greedy decoding means:
- at each step, choose the token with the **highest probability**
- feed that prediction back into the decoder
- continue until you have a short output sentence

Because this is a toy example, the translation may not be perfect.  
The purpose is to understand the **step-by-step generation flow**.

After this, you will do the same for **English -> Spanish**.


In [20]:
#  greedy decoding with a tiny hand-made probability table (English -> Dutch)
toy_vocab = {1: "<SOS>", 2: "ik", 3: "ben", 4: "blij", 5: "verdrietig"}

# Pretend these are the model's best predictions at each step
predicted_ids = [2, 3, 4]

generated_words = []
current_token = 1  # <SOS>

print("Source sentence: i am happy")
print("Expected language: Dutch")
print("Start token:", toy_vocab[current_token])

for step, next_id in enumerate(predicted_ids, start=1):
    print(f"Step {step}: current token = {toy_vocab[current_token]} -> predict {toy_vocab[next_id]}")
    generated_words.append(toy_vocab[next_id])
    current_token = next_id

print("\nGreedy decoded sentence:", " ".join(generated_words))


Source sentence: i am happy
Expected language: Dutch
Start token: <SOS>
Step 1: current token = <SOS> -> predict ik
Step 2: current token = ik -> predict ben
Step 3: current token = ben -> predict blij

Greedy decoded sentence: ik ben blij


## Task 4 — Greedy decoding on a toy example

Your goal is to generate an **English -> Spanish** translation one token at a time.

Greedy decoding means:
- at each step, choose the word with the highest predicted probability

### What to do
1. Use the trained/initialized encoder and decoder
2. Encode the sentence `"i am happy"`
3. Start the decoder with `<SOS>`
4. Predict 3 output tokens
5. Convert predicted IDs back to words

> This is a toy example, so the output may not be good before training.  
> The purpose is to understand the decoding process.


In [ ]:

test_sentence = "i am happy"
src_ids = encode_src(test_sentence)
src_ids = src_ids + [0] * (max_src - len(src_ids))
src_tensor = torch.tensor([src_ids], dtype=torch.long).to(device)

with torch.no_grad():
    # TODO: encode the source sentence
    enc_outputs, hidden = None, None

    # start with SOS
    current_token = torch.tensor([[1]], dtype=torch.long).to(device)

    predicted_words = []
    for step in range(3):
        # TODO: run one decoder step
        logits, hidden = None, None

        # TODO: choose the most likely token
        pred_id = None

        if pred_id in idx2tgt:
            predicted_words.append(idx2tgt[pred_id])
        else:
            predicted_words.append("<UNK>")

        current_token = torch.tensor([[pred_id]], dtype=torch.long).to(device)

print("Source:", test_sentence)
print("Predicted target tokens:", predicted_words)



## Optional reflection questions

1. Why do we need **padding** for batched sequence models?
2. In a Seq2Seq model, what is the role of the **encoder hidden state**?
3. Why might greedy decoding fail compared with beam search?
4. Why is machine translation harder than next-word prediction on a tiny toy corpus?
